# PLAYHACK — Exploratory Data Analysis & Pipeline Inspection

This notebook demonstrates:
1. Loading and inspecting raw multi-source wearable and session records.
2. Validating temporal boundaries (zero leakage from Days 31–60).
3. Feature engineering and correlation with injury risk.
4. Athlete workload, sleep, and heart rate trend inspection.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import RawDataLoader
from src.feature_engineering import combine_features
from src.leakage_guard import leakage_guard

## 1. Load Raw Datasets

In [ ]:
loader = RawDataLoader('../data')
meta_df = loader.load_athlete_metadata()
labels_df = loader.load_train_labels()
act_df = loader.load_daily_activity()
sleep_df = loader.load_sleep_day()

print('Athlete Metadata:', meta_df.shape)
print('Training Labels:', labels_df.shape)
print('Daily Activity:', act_df.shape)
print('Sleep Records:', sleep_df.shape)

## 2. Inspect Target Class Balance (Task A)

In [ ]:
injury_counts = labels_df['injured_in_risk_window'].value_counts()
print(injury_counts)

plt.figure(figsize=(6, 4))
sns.barplot(x=injury_counts.index, y=injury_counts.values, palette='viridis')
plt.title('Distribution of Target (injured_in_risk_window)')
plt.xlabel('Injured (0 = No, 1 = Yes)')
plt.ylabel('Number of Athletes')
plt.show()

## 3. Load or Build Extracted Feature Dataset

In [ ]:
import os
features_path = '../outputs/features.parquet'
if os.path.exists(features_path):
    features_df = pd.read_parquet(features_path)
    print('Loaded features:', features_df.shape)
else:
    print('Features file not yet built. Run feature_engineering.py to generate.')

## 4. Workload Spike vs Injury Correlation

In [ ]:
if 'features_df' in locals() and 'training_load_change_7d_vs_30d' in features_df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(
        data=features_df,
        x='injured_in_risk_window',
        y='training_load_change_7d_vs_30d',
        palette='Set2'
    )
    plt.title('Training Load Change (7d vs 30d) by Injury Status')
    plt.xlabel('Injured in Risk Window')
    plt.ylabel('Workload Ratio (ACWR Proxy)')
    plt.show()